# Experiment: No-Lab Execution Ladder

Objective:
- Rank the most useful actions Nakafa Lab can take before a wet-lab partner is available.
- Keep patient-specific claims blocked unless a clinician reviews the packet.
- Separate practical no-lab work from therapy recommendations.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    """No-lab action scored by practical and safety value."""

    name: str
    evidence_value: int
    urgency: int
    privacy_risk: int
    clinician_value: int
    lab_dependency: int
    treatment_risk: int

    @property
    def score(self) -> int:
        """Return the no-lab priority score."""
        return (
            self.evidence_value
            + self.urgency
            + self.clinician_value
            - self.privacy_risk
            - self.lab_dependency
            - self.treatment_risk
        )

## Scoring Model

Higher scores mean a no-lab action is more useful now. Privacy risk, lab dependency, and treatment-risk penalties keep the ladder away from unsafe shortcuts.


In [ ]:
actions = [
    Action("minimum_hematologist_packet", 5, 5, 2, 5, 0, 0),
    Action("private_record_index", 5, 5, 1, 4, 0, 0),
    Action("transfusion_burden_calculation", 5, 5, 2, 5, 0, 0),
    Action("iron_chelation_organ_risk_packet", 5, 4, 2, 5, 0, 0),
    Action("immune_blood_bank_packet", 5, 4, 2, 5, 0, 0),
    Action("advanced_therapy_referral_readiness", 4, 4, 2, 5, 0, 0),
    Action("current_trial_watchlist_refresh", 4, 3, 0, 3, 0, 0),
    Action("computational_candidate_rerank", 3, 2, 0, 2, 0, 1),
    Action("lab_quote_request_package", 3, 2, 0, 3, 2, 0),
    Action("new_candidate_claim", 2, 1, 0, 0, 3, 5),
]

ranking = sorted(
    (
        {
            "action": action.name,
            "score": action.score,
            "requires_lab": action.lab_dependency > 0,
            "patient_claim_allowed": False,
        }
        for action in actions
    ),
    key=lambda item: item["score"],
    reverse=True,
)
ranking

## Gate Decision

The no-lab ladder should favor record readiness, clinician review, and referral screening before more candidate claims.


In [ ]:
top_actions = [item["action"] for item in ranking[:5]]
blocked_claims = [item for item in ranking if item["action"] == "new_candidate_claim"]

decision = {
    "top_actions": top_actions,
    "next_operational_label": "no_lab_execution_ladder_active",
    "patient_specific_claims": "blocked_until_clinician_review",
    "lowest_ranked_action": ranking[-1]["action"],
    "new_candidate_claim_score": blocked_claims[0]["score"],
}
decision